## Regras

### Estação empresarial/móvel: 

- Campo 'alias_ebt' preenchimento: empresarial/móvel
- Campo 'alias_ebt' sem preenchimento: móvel

### Estações Oi: 

Todo site móvel

### Geradores

- Mais de uma ocorrência de geradores ativos: redundante 
- Uma ocorrência: singelo
- Sem ocorrência: sem gerador

### Hierarquia Estação

- 60: Estações de ponta ou repetidores RF
- 70: Estações estratégicas e monosites
- 80: Estações concentradores com repetidores MW<20 sites
- 90: Estações com roteadores IpRAN ou com repetidores MW>=20 sites
- 100: Prédios de centrais, BSC e RNC Remotas e GPON

In [2]:
import pandas as pd

In [3]:
map = {
    60: 4, #Estações de ponta ou repetidores RF
    70: 4, #Estações estratégicas e Monosites
    80: 4, #Estações concentradores com repetidores MW<20sites
    90: 4,  #Estações com roteadores IpRAN ou com repetidores MW>=20 sites
    100: 4  #Predios de centrais, BSC e RNC Remotas e GPON
}

In [4]:
df_indisponibilidade = pd.read_pickle(r'C:\Users\f252191\gerenciador_de_baterias\backend\data\raw\indisponibilidade.pkl')
df_indisponibilidade = df_indisponibilidade[['estacao', 'mttr']]
df_indisponibilidade = df_indisponibilidade[df_indisponibilidade['estacao'] != '']
df_indisponibilidade['mttr_maior_que_2h'] = (df_indisponibilidade['mttr'] > 7200).map({True: 'SIM', False: 'NÃO'})
df_indisponibilidade = df_indisponibilidade[['estacao', 'mttr_maior_que_2h']]
df_indisponibilidade = df_indisponibilidade.drop_duplicates()
df_indisponibilidade.head()

,estacao,mttr_maior_que_2h
0,DFPADR4,SIM
2,DFGMAR1,NÃO
3,DFGMA81,SIM
5,DFGMA32,NÃO
7,CESOL03,SIM


In [5]:
pd.read_pickle(r'C:\Users\f252191\gerenciador_de_baterias\backend\data\raw\estacoes.pkl')

,estacao,cluster,uf,municipio,ibge,alias_ebt,nome_ref_estacao,severidade_omr,finalidade_movel,finalidade_empresarial,finalidade_residencial,latitude,longitude
0,BAFSA37,BA/SE,BA,FEIRA DE SANTANA,2910800,FSA VX,,04 - MODERADO,SITE RF,PPC,,-12.253807,-38.964660
1,BAFSA37_001,BA/SE,BA,FEIRA DE SANTANA,2910800,,SLS - FEIRA DE SANTANA,,SITE RF,,,-12.253044,-38.966417
2,BAFSA37_002,BA/SE,BA,FEIRA DE SANTANA,2910800,,SLS - FEIRA DE SANTANA,,SETOR REMOTO,,,-12.255990,-38.965210
3,BAFSA37_003,BA/SE,BA,FEIRA DE SANTANA,2910800,,,,SITE RF,,,-12.252900,-38.966290
4,BAFSA38,BA/SE,BA,FEIRA DE SANTANA,2910800,FSA SC5,,16 - CRITICO,SITE RF,TER,,-12.256380,-38.969304
...,...,...,...,...,...,...,...,...,...,...,...,...,...
36177,BAFSA32_002,BA/SE,BA,FEIRA DE SANTANA,2910800,,,,SITE RF,,,-12.257820,-38.958350
36178,BAFSA33,BA/SE,BA,FEIRA DE SANTANA,2910800,,,08 - SEVERO,SITE RF,,,-12.271640,-38.945920
36179,BAFSA34,BA/SE,BA,FEIRA DE SANTANA,2910800,,,04 - MODERADO,SITE RF,,,-12.246166,-38.959333
36180,BAFSA35,BA/SE,BA,FEIRA DE SANTANA,2910800,,,04 - MODERADO,SITE RF,,,-12.237776,-38.970623


In [6]:
df_estacoes = pd.read_pickle(r'C:\Users\f252191\gerenciador_de_baterias\backend\data\raw\estacoes.pkl')

df_estacoes['tipo_estacao'] = (
    df_estacoes['alias_ebt'] != ''
).map({
    True: 'Empresarial/Móvel',
    False: 'Móvel'
})

df_estacoes['qtd_estacoes_municipio'] = (
    df_estacoes.groupby('municipio')['estacao']
    .transform('nunique')
)

df_estacoes['municipio_ate_5_estacoes'] = (
    df_estacoes['qtd_estacoes_municipio'] <= 5
).map({
    True: 'SIM',
    False: 'NÃO'
})

# seleciona colunas finais
df_estacoes = df_estacoes[[
    'estacao', 
    'tipo_estacao', 
    'municipio_ate_5_estacoes'
]]

df_estacoes.head()

,estacao,tipo_estacao,municipio_ate_5_estacoes
0,BAFSA37,Empresarial/Móvel,NÃO
1,BAFSA37_001,Móvel,NÃO
2,BAFSA37_002,Móvel,NÃO
3,BAFSA37_003,Móvel,NÃO
4,BAFSA38,Empresarial/Móvel,NÃO


In [7]:
df_pontuacao_hierarquia = pd.read_pickle(r'C:\Users\f252191\gerenciador_de_baterias\backend\data\aggregated\pontuacao_hierarquia.pkl')
df_pontuacao_hierarquia.head()

,estacao,pontuacao_hierarquia
0,BA00915,60.0
1,BA00887,60.0
2,BA01138,60.0
3,BAAAE01,70.0
4,BAAAE02,90.0


In [8]:
df_pontuacao_hierarquia.info()

<class 'pandas.DataFrame'>
RangeIndex: 36182 entries, 0 to 36181
Data columns (total 2 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   estacao               36182 non-null  object 
 1   pontuacao_hierarquia  29948 non-null  float64
dtypes: float64(1), object(1)
memory usage: 565.5+ KB


In [9]:
df_geradores = pd.read_pickle(r'C:\Users\f252191\gerenciador_de_baterias\backend\data\raw\geradores.pkl')
df_geradores.head()

,estacao,tipo_gerador
0,RJOFRG,REDUNDANTE
1,RJOBOT,REDUNDANTE
2,RJOCPG,REDUNDANTE
3,RJOPRN,SEM GERADOR
4,RJOPIA,REDUNDANTE


In [11]:
df = df_estacoes.merge(df_pontuacao_hierarquia, on='estacao', how='left')
df = df.merge(df_indisponibilidade, on='estacao', how='left')
df = df.merge(df_geradores, on='estacao', how='left')
#nulos
df['pontuacao_hierarquia'] = df['pontuacao_hierarquia'].fillna(60)
df['mttr_maior_que_2h'] = df['mttr_maior_que_2h'].fillna('SEM INDISPONIBILIDADE')
df['tipo_gerador'] = df['tipo_gerador'].fillna('SEM GERADOR')
df['cow'] = ''
df['autonomia_projetada'] = ''
df

,estacao,tipo_estacao,municipio_ate_5_estacoes,pontuacao_hierarquia,mttr_maior_que_2h,tipo_gerador,cow,autonomia_projetada
0,BAFSA37,Empresarial/Móvel,NÃO,70.0,NÃO,SEM GERADOR,,
1,BAFSA37,Empresarial/Móvel,NÃO,70.0,SIM,SEM GERADOR,,
2,BAFSA37_001,Móvel,NÃO,60.0,SEM INDISPONIBILIDADE,SEM GERADOR,,
3,BAFSA37_002,Móvel,NÃO,60.0,SEM INDISPONIBILIDADE,SEM GERADOR,,
4,BAFSA37_003,Móvel,NÃO,60.0,SEM INDISPONIBILIDADE,SEM GERADOR,,
...,...,...,...,...,...,...,...,...
61107,BAFSA34,Móvel,NÃO,70.0,SIM,SEM GERADOR,,
61108,BAFSA35,Móvel,NÃO,70.0,SIM,SEM GERADOR,,
61109,BAFSA35,Móvel,NÃO,70.0,NÃO,SEM GERADOR,,
61110,BAFSA36,Empresarial/Móvel,NÃO,90.0,NÃO,SEM GERADOR,,
